# Training with Segura dataset

```
# Ce texte est au format code
```



**LTSM model**

In [8]:
# ==============================================================================
# 1. SETUP & LIBRARIES
# ==============================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from google.colab import files

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================================================================
# 2. UPLOAD DATASET & PHYSICAL FEATURE ENGINEERING
# ==============================================================================
print("Please upload your un-augmented Spain CSV dataset (without SMOTE):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

# Sort chronologically by date
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# 1. Tank Fullness Feature: Inverse of distance (lower distance = fuller tank)
df['tank_fullness'] = 1.0 / (df['distance_cm'] + 1e-5)

# 2. Rain Dynamics: Rolling sum to capture accumulated rainfall driving flow
df['rain_rolling_sum_6'] = df['rain_raw'].rolling(window=6, min_periods=1).sum()

# Define all input features including physical indicators
feature_cols = [
    'flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage',
    'tank_fullness', 'rain_rolling_sum_6'
]

# Targets to forecast 12 steps into the future
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

# Normalize features to [0, 1] range
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[feature_cols])

# Train / Test split (80% train, 20% test chronologically)
train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

# ==============================================================================
# 3. MULTI-TARGET SLIDING SEQUENCE DATASET
# ==============================================================================
class MultiTargetSequenceDataset(Dataset):
    def __init__(self, data, target_indices, seq_len=36, forecast_len=12):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.seq_len = seq_len
        self.forecast_len = forecast_len
        self.target_indices = target_indices

    def __len__(self):
        return len(self.data) - self.seq_len - self.forecast_len + 1

    def __getitem__(self, idx):
        # Input (X): 36 historical sequence steps across all features
        x = self.data[idx : idx + self.seq_len]

        # Target (Y): Next 12 future steps for target features
        # Shape: (forecast_len, num_targets)
        y = self.data[
            idx + self.seq_len : idx + self.seq_len + self.forecast_len,
            self.target_indices
        ]
        return x, y

SEQ_LEN = 36        # Lookback window (36 states)
FORECAST_LEN = 12   # Forecast horizon (next 12 states)

train_dataset = MultiTargetSequenceDataset(train_data, target_indices, SEQ_LEN, FORECAST_LEN)
test_dataset = MultiTargetSequenceDataset(test_data, target_indices, SEQ_LEN, FORECAST_LEN)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ==============================================================================
# 4. PYTORCH MULTI-OUTPUT LSTM MODEL
# ==============================================================================
class CloudMultiTargetLSTM(nn.Module):
    def __init__(self, input_dim, num_targets, hidden_dim=128, num_layers=2, forecast_len=12):
        super().__init__()
        self.forecast_len = forecast_len
        self.num_targets = num_targets

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        self.fc = nn.Linear(hidden_dim, forecast_len * num_targets)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :] # Take last hidden state

        preds = self.fc(last_hidden)
        # Reshape output to (batch_size, forecast_len, num_targets)
        preds = preds.view(-1, self.forecast_len, self.num_targets)
        return preds

model = CloudMultiTargetLSTM(
    input_dim=len(feature_cols),
    num_targets=len(target_cols),
    forecast_len=FORECAST_LEN
).to(device)

# ==============================================================================
# 5. TRAINING LOOP WITH HUBER LOSS & EARLY STOPPING
# ==============================================================================
# Huber Loss handles erratic flow rate spikes without over-penalizing outliers
criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 40
best_loss = float('inf')
patience = 7
patience_counter = 0

print("\nStarting Training...")
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation Check
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            preds = model(x_batch)
            val_loss += criterion(preds, y_batch).item()

    avg_val_loss = val_loss / len(test_loader)

    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")

    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model.state_dict(), "best_cloud_lstm.pth")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at Epoch {epoch+1}")
            break

# ==============================================================================
# 6. EVALUATION & METRIC CALCULATION
# ==============================================================================
model.load_state_dict(torch.load("best_cloud_lstm.pth"))
model.eval()

all_preds, all_targets = [], []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        preds = model(x_batch)

        all_preds.append(preds.cpu().numpy())
        all_targets.append(y_batch.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

print("\n--- TEST EVALUATION METRICS (PHYSICAL UNITS) ---")
for i, col in enumerate(target_cols):
    t_idx = target_indices[i]

    # Inverse transform predictions back to real scale
    pred_dummies = np.zeros((all_preds.shape[0] * FORECAST_LEN, len(feature_cols)))
    target_dummies = np.zeros((all_targets.shape[0] * FORECAST_LEN, len(feature_cols)))

    pred_dummies[:, t_idx] = all_preds[:, :, i].flatten()
    target_dummies[:, t_idx] = all_targets[:, :, i].flatten()

    real_pred = scaler.inverse_transform(pred_dummies)[:, t_idx]
    real_target = scaler.inverse_transform(target_dummies)[:, t_idx]

    mae = np.mean(np.abs(real_pred - real_target))
    rmse = np.sqrt(np.mean((real_pred - real_target) ** 2))
    target_mean = real_target.mean()
    relative_err = (mae / target_mean) * 100

    print(f"Target Feature: {col}")
    print(f"  MAE : {mae:.3f}")
    print(f"  RMSE: {rmse:.3f}")
    print(f"  Mean Value: {target_mean:.2f}")
    print(f"  Error Percentage: {relative_err:.2f}%\n")


Using device: cuda
Please upload your un-augmented Spain CSV dataset (without SMOTE):


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (2).csv

Starting Training...
Epoch 01/40 | Train Loss: 0.01421 | Val Loss: 0.01568
Epoch 02/40 | Train Loss: 0.01034 | Val Loss: 0.01436
Epoch 03/40 | Train Loss: 0.00987 | Val Loss: 0.01354
Epoch 04/40 | Train Loss: 0.00965 | Val Loss: 0.01278
Epoch 05/40 | Train Loss: 0.00949 | Val Loss: 0.01225
Epoch 06/40 | Train Loss: 0.00936 | Val Loss: 0.01180
Epoch 07/40 | Train Loss: 0.00924 | Val Loss: 0.01137
Epoch 08/40 | Train Loss: 0.00911 | Val Loss: 0.01083
Epoch 09/40 | Train Loss: 0.00894 | Val Loss: 0.01004
Epoch 10/40 | Train Loss: 0.00876 | Val Loss: 0.00908
Epoch 11/40 | Train Loss: 0.00859 | Val Loss: 0.00849
Epoch 12/40 | Train Loss: 0.00857 | Val Loss: 0.00839
Epoch 13/40 | Train Loss: 0.00848 | Val Loss: 0.00813
Epoch 14/40 | Train Loss: 0.00843 | Val Loss: 0.00807
Epoch 15/40 | Train Loss: 0.00840 | Val Loss: 0.00802
Epoch 16/40 | Train Loss: 0.00840 | Val Loss: 0.00798
Epoch 17/40 | Train Loss: 0.

In [9]:
# ==============================================================================
# 7. DOWNLOAD WEIGHTS FILE FOR CLOUD DEPLOYMENT
# ==============================================================================
files.download("best_cloud_lstm.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**CNN-LSTM hybrid model**

In [10]:
# ==============================================================================
# 1. SETUP & LIBRARIES
# ==============================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from google.colab import files

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================================================================
# 2. UPLOAD DATASET & PHYSICAL FEATURE ENGINEERING
# ==============================================================================
print("Please upload your un-augmented Spain CSV dataset (without SMOTE):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

# Sort chronologically by date
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# 1. Tank Fullness Feature: Inverse of distance (lower distance = fuller tank)
df['tank_fullness'] = 1.0 / (df['distance_cm'] + 1e-5)

# 2. Rain Dynamics: Rolling sum to capture accumulated rainfall driving flow
df['rain_rolling_sum_6'] = df['rain_raw'].rolling(window=6, min_periods=1).sum()

# Define feature columns
feature_cols = [
    'flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage',
    'tank_fullness', 'rain_rolling_sum_6'
]

# Targets to forecast 12 steps into the future
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

# Normalize features to [0, 1] range
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[feature_cols])

# Train / Test split (80% train, 20% test chronologically)
train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

# ==============================================================================
# 3. MULTI-TARGET SLIDING SEQUENCE DATASET
# ==============================================================================
class MultiTargetSequenceDataset(Dataset):
    def __init__(self, data, target_indices, seq_len=36, forecast_len=12):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.seq_len = seq_len
        self.forecast_len = forecast_len
        self.target_indices = target_indices

    def __len__(self):
        return len(self.data) - self.seq_len - self.forecast_len + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[
            idx + self.seq_len : idx + self.seq_len + self.forecast_len,
            self.target_indices
        ]
        return x, y

SEQ_LEN = 36        # Lookback window (36 states)
FORECAST_LEN = 12   # Forecast horizon (next 12 states)

train_dataset = MultiTargetSequenceDataset(train_data, target_indices, SEQ_LEN, FORECAST_LEN)
test_dataset = MultiTargetSequenceDataset(test_data, target_indices, SEQ_LEN, FORECAST_LEN)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ==============================================================================
# 4. 1D CNN + LSTM HYBRID ARCHITECTURE
# ==============================================================================
class CNNLSTMHybrid(nn.Module):
    def __init__(self, input_dim, num_targets, hidden_dim=128, num_layers=2, forecast_len=12):
        super().__init__()
        self.forecast_len = forecast_len
        self.num_targets = num_targets

        # 1D Convolutional Layer for extracting local burst patterns
        # Note: Conv1d expects shape (batch_size, channels/features, seq_len)
        self.conv1d = nn.Conv1d(
            in_channels=input_dim,
            out_channels=64,
            kernel_size=3,
            padding=1
        )
        self.relu = nn.ReLU()

        # LSTM processes spatial feature maps over time
        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.fc = nn.Linear(hidden_dim, forecast_len * num_targets)

    def forward(self, x):
        # x shape: (batch_size, seq_len, features) -> Permute to (batch_size, features, seq_len)
        x_cnn = x.permute(0, 2, 1)
        x_cnn = self.relu(self.conv1d(x_cnn))

        # Permute back to (batch_size, seq_len, cnn_out_channels) for LSTM
        x_lstm_in = x_cnn.permute(0, 2, 1)

        out, _ = self.lstm(x_lstm_in)
        last_hidden = out[:, -1, :]

        preds = self.fc(last_hidden)
        preds = preds.view(-1, self.forecast_len, self.num_targets)
        return preds

model = CNNLSTMHybrid(
    input_dim=len(feature_cols),
    num_targets=len(target_cols),
    forecast_len=FORECAST_LEN
).to(device)

# ==============================================================================
# 5. TRAINING LOOP WITH HUBER LOSS & EARLY STOPPING
# ==============================================================================
criterion = nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 40
best_loss = float('inf')
patience = 7
patience_counter = 0

print("\nStarting Training (CNN-LSTM Hybrid)...")
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation Check
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            preds = model(x_batch)
            val_loss += criterion(preds, y_batch).item()

    avg_val_loss = val_loss / len(test_loader)

    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")

    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model.state_dict(), "best_cnn_lstm_hybrid.pth")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at Epoch {epoch+1}")
            break

# ==============================================================================
# 6. EVALUATION & METRIC CALCULATION
# ==============================================================================
model.load_state_dict(torch.load("best_cnn_lstm_hybrid.pth"))
model.eval()

all_preds, all_targets = [], []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        preds = model(x_batch)

        all_preds.append(preds.cpu().numpy())
        all_targets.append(y_batch.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

print("\n--- TEST EVALUATION METRICS (CNN-LSTM HYBRID) ---")
for i, col in enumerate(target_cols):
    t_idx = target_indices[i]

    pred_dummies = np.zeros((all_preds.shape[0] * FORECAST_LEN, len(feature_cols)))
    target_dummies = np.zeros((all_targets.shape[0] * FORECAST_LEN, len(feature_cols)))

    pred_dummies[:, t_idx] = all_preds[:, :, i].flatten()
    target_dummies[:, t_idx] = all_targets[:, :, i].flatten()

    real_pred = scaler.inverse_transform(pred_dummies)[:, t_idx]
    real_target = scaler.inverse_transform(target_dummies)[:, t_idx]

    mae = np.mean(np.abs(real_pred - real_target))
    rmse = np.sqrt(np.mean((real_pred - real_target) ** 2))
    target_mean = real_target.mean()
    relative_err = (mae / target_mean) * 100

    print(f"Target Feature: {col}")
    print(f"  MAE : {mae:.3f}")
    print(f"  RMSE: {rmse:.3f}")
    print(f"  Mean Value: {target_mean:.2f}")
    print(f"  Error Percentage: {relative_err:.2f}%\n")

# ==============================================================================
# 7. DOWNLOAD WEIGHTS FILE FOR CLOUD DEPLOYMENT
# ==============================================================================
files.download("best_cnn_lstm_hybrid.pth")

Using device: cuda
Please upload your un-augmented Spain CSV dataset (without SMOTE):


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (3).csv

Starting Training (CNN-LSTM Hybrid)...
Epoch 01/40 | Train Loss: 0.01355 | Val Loss: 0.01456
Epoch 02/40 | Train Loss: 0.01016 | Val Loss: 0.01364
Epoch 03/40 | Train Loss: 0.00972 | Val Loss: 0.01297
Epoch 04/40 | Train Loss: 0.00948 | Val Loss: 0.01218
Epoch 05/40 | Train Loss: 0.00923 | Val Loss: 0.01125
Epoch 06/40 | Train Loss: 0.00896 | Val Loss: 0.01041
Epoch 07/40 | Train Loss: 0.00872 | Val Loss: 0.00940
Epoch 08/40 | Train Loss: 0.00853 | Val Loss: 0.00866
Epoch 09/40 | Train Loss: 0.00843 | Val Loss: 0.00833
Epoch 10/40 | Train Loss: 0.00837 | Val Loss: 0.00826
Epoch 11/40 | Train Loss: 0.00834 | Val Loss: 0.00817
Epoch 12/40 | Train Loss: 0.00831 | Val Loss: 0.00803
Epoch 13/40 | Train Loss: 0.00827 | Val Loss: 0.00809
Epoch 14/40 | Train Loss: 0.00826 | Val Loss: 0.00788
Epoch 15/40 | Train Loss: 0.00825 | Val Loss: 0.00776
Epoch 16/40 | Train Loss: 0.00821 | Val Loss: 0.00787
Epoch 17/4

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**XGBOOST model for Flow rate column**

In [12]:
# ==============================================================================
# 1. SETUP & LIBRARIES
# ==============================================================================
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from google.colab import files

# ==============================================================================
# 2. UPLOAD DATASET & FEATURE ENGINEERING
# ==============================================================================
print("Please upload your un-augmented Spain CSV dataset (without SMOTE):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

# Sort chronologically by date
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Feature engineering matching previous setup
df['tank_fullness'] = 1.0 / (df['distance_cm'] + 1e-5)
df['rain_rolling_sum_6'] = df['rain_raw'].rolling(window=6, min_periods=1).sum()

feature_cols = [
    'flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage',
    'tank_fullness', 'rain_rolling_sum_6'
]
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

# Normalize features
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[feature_cols])

# Chronological Train / Test Split (80/20)
train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

# ==============================================================================
# 3. CONVERT TIME-SERIES WINDOWS TO TABULAR NUMPY ARRAYS
# ==============================================================================
SEQ_LEN = 36
FORECAST_LEN = 12

def create_windowed_numpy(data, target_indices, seq_len=36, forecast_len=12):
    X, Y = [], []
    for i in range(len(data) - seq_len - forecast_len + 1):
        X.append(data[i : i + seq_len])
        Y.append(data[i + seq_len : i + seq_len + forecast_len, target_indices])
    return np.array(X), np.array(Y)

X_train, Y_train = create_windowed_numpy(train_data, target_indices, SEQ_LEN, FORECAST_LEN)
X_test, Y_test = create_windowed_numpy(test_data, target_indices, SEQ_LEN, FORECAST_LEN)

# Flatten 3D sequence inputs (samples, 36, features) -> 2D tabular (samples, 36 * features)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# ==============================================================================
# 4. TRAIN & EVALUATE MULTI-OUTPUT XGBOOST
# ==============================================================================
print("\n--- TRAINING MULTI-OUTPUT XGBOOST MODEL ---")

for i, col in enumerate(target_cols):
    t_idx = target_indices[i]

    # Target values for this specific feature across all 12 future steps
    y_train_target = Y_train[:, :, i]  # Shape: (samples, 12)
    y_test_target = Y_test[:, :, i]    # Shape: (samples, 12)

    # Fit XGBoost Multi-Output Regressor
    model = xgb.XGBRegressor(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    model.fit(X_train_flat, y_train_target)

    # Predict next 12 steps
    preds_scaled = model.predict(X_test_flat)

    # Unscale predictions and ground truth back to physical units
    pred_dummies = np.zeros((preds_scaled.shape[0] * FORECAST_LEN, len(feature_cols)))
    target_dummies = np.zeros((y_test_target.shape[0] * FORECAST_LEN, len(feature_cols)))

    pred_dummies[:, t_idx] = preds_scaled.flatten()
    target_dummies[:, t_idx] = y_test_target.flatten()

    real_pred = scaler.inverse_transform(pred_dummies)[:, t_idx]
    real_target = scaler.inverse_transform(target_dummies)[:, t_idx]

    mae = mean_absolute_error(real_target, real_pred)
    rmse = np.sqrt(mean_squared_error(real_target, real_pred))
    target_mean = real_target.mean()
    relative_err = (mae / target_mean) * 100

    print(f"Target Feature: {col}")
    print(f"  MAE : {mae:.3f}")
    print(f"  RMSE: {rmse:.3f}")
    print(f"  Mean Value: {target_mean:.2f}")
    print(f"  Error Percentage: {relative_err:.2f}%\n")

Please upload your un-augmented Spain CSV dataset (without SMOTE):


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (4).csv

--- TRAINING MULTI-OUTPUT XGBOOST MODEL ---
Target Feature: distance_cm
  MAE : 2.860
  RMSE: 6.886
  Mean Value: 25.75
  Error Percentage: 11.11%

Target Feature: soil_raw
  MAE : 247.206
  RMSE: 625.908
  Mean Value: 3790.68
  Error Percentage: 6.52%

Target Feature: flow_rate_l_min
  MAE : 25.973
  RMSE: 68.057
  Mean Value: 38.43
  Error Percentage: 67.59%



**Solution: Two-Stage Classification + Regression Pipeline**
Instead of forcing a single regression model to predict both zero-flow states and discharge magnitudes, split the problem into a two-step pipeline:

Telemetry Input (36 Steps)
         │
         ▼
[ Stage 1: XGBoost Classifier ] ──> Is Flow Active? (0 = No Flow, 1 = Flowing)
         │
         ├── If 0 ──> Output: 0.00 L/min
         │
         └── If 1 ──> [ Stage 2: XGBoost Regressor ] ──> Output Predicted L/min Magni
        

**XGB with ignoring the risk value 3**

In [15]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Please upload your un-augmented Spain CSV dataset (without SMOTE):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

# Exclude flood_risk_level == 3 (hardware error codes handled upstream by Layer 2)
df_clean = df[df['flood_risk_level'] != 3].copy().reset_index(drop=True)

# 2. Feature Engineering
df_clean['tank_fullness'] = 1.0 / (df_clean['distance_cm'] + 1e-5)
df_clean['rain_rolling_sum_6'] = df_clean['rain_raw'].rolling(window=6, min_periods=1).sum()

feature_cols = ['flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage', 'tank_fullness', 'rain_rolling_sum_6']
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

# 3. Normalization
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_clean[feature_cols])

train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

# 4. Windowing Function (36 Steps Input -> 12 Steps Multi-Step Output)
SEQ_LEN = 36
FORECAST_LEN = 12

def create_windowed_numpy(data, target_indices, seq_len=36, forecast_len=12):
    X, Y = [], []
    for i in range(len(data) - seq_len - forecast_len + 1):
        X.append(data[i : i + seq_len])
        Y.append(data[i + seq_len : i + seq_len + forecast_len, target_indices])
    return np.array(X), np.array(Y)

X_train, Y_train = create_windowed_numpy(train_data, target_indices, SEQ_LEN, FORECAST_LEN)
X_test, Y_test = create_windowed_numpy(test_data, target_indices, SEQ_LEN, FORECAST_LEN)

# Flatten input windows for XGBoost: (Samples, 36 * Features)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# 5. Model Training & Evaluation
print("--- TRAINING LAYER 3 MULTI-OUTPUT XGBOOST MODEL ---")

for i, col in enumerate(target_cols):
    t_idx = target_indices[i]

    y_train_target = Y_train[:, :, i]  # Shape: (samples, 12)
    y_test_target = Y_test[:, :, i]    # Shape: (samples, 12)

    # Train multi-output XGBoost Regressor
    model = xgb.XGBRegressor(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    model.fit(X_train_flat, y_train_target)
    preds_scaled = model.predict(X_test_flat)

    # Unscale predictions back to real physical units
    pred_dummies = np.zeros((preds_scaled.shape[0] * FORECAST_LEN, len(feature_cols)))
    target_dummies = np.zeros((y_test_target.shape[0] * FORECAST_LEN, len(feature_cols)))

    pred_dummies[:, t_idx] = preds_scaled.flatten()
    target_dummies[:, t_idx] = y_test_target.flatten()

    real_pred = scaler.inverse_transform(pred_dummies)[:, t_idx]
    real_target = scaler.inverse_transform(target_dummies)[:, t_idx]

    # Calculate Metrics
    mae = mean_absolute_error(real_target, real_pred)
    rmse = np.sqrt(mean_squared_error(real_target, real_pred))
    target_mean = real_target.mean()
    error_pct = (mae / target_mean) * 100

    print(f"Target Feature: {col}")
    print(f"  MAE : {mae:.3f}")
    print(f"  RMSE: {rmse:.3f}")
    print(f"  Mean Value: {target_mean:.2f}")
    print(f"  Error Percentage: {error_pct:.2f}%\n")

Please upload your un-augmented Spain CSV dataset (without SMOTE):


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (5).csv
--- TRAINING LAYER 3 MULTI-OUTPUT XGBOOST MODEL ---
Target Feature: distance_cm
  MAE : 0.368
  RMSE: 0.724
  Mean Value: 26.59
  Error Percentage: 1.38%

Target Feature: soil_raw
  MAE : 38.402
  RMSE: 122.678
  Mean Value: 3865.71
  Error Percentage: 0.99%

Target Feature: flow_rate_l_min
  MAE : 5.145
  RMSE: 13.091
  Mean Value: 30.31
  Error Percentage: 16.97%



*Here Am going to retrain just for download purposes*

In [24]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from google.colab import files
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Please upload your un-augmented Spain CSV dataset (without SMOTE):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

# 1. Exclude flood_risk_level == 3 (hardware error codes handled upstream by Layer 2)
df_clean = df[df['flood_risk_level'] != 3].copy().reset_index(drop=True)

# 2. Feature Engineering
df_clean['tank_fullness'] = 1.0 / (df_clean['distance_cm'] + 1e-5)
df_clean['rain_rolling_sum_6'] = df_clean['rain_raw'].rolling(window=6, min_periods=1).sum()

feature_cols = ['flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage', 'tank_fullness', 'rain_rolling_sum_6']
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

# 3. Normalization
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_clean[feature_cols])

# Save and export the fitted scaler for inference deployment
scaler_path = 'scaler_xgboost.pkl'
joblib.dump(scaler, scaler_path)

train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

# 4. Windowing Function (36 Steps Input -> 12 Steps Multi-Step Output)
SEQ_LEN = 36
FORECAST_LEN = 12

def create_windowed_numpy(data, target_indices, seq_len=36, forecast_len=12):
    X, Y = [], []
    for i in range(len(data) - seq_len - forecast_len + 1):
        X.append(data[i : i + seq_len])
        Y.append(data[i + seq_len : i + seq_len + forecast_len, target_indices])
    return np.array(X), np.array(Y)

X_train, Y_train = create_windowed_numpy(train_data, target_indices, SEQ_LEN, FORECAST_LEN)
X_test, Y_test = create_windowed_numpy(test_data, target_indices, SEQ_LEN, FORECAST_LEN)

# Flatten input windows for XGBoost: (Samples, 36 * Features)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# 5. Model Training, Saving & Evaluation
print("\n--- TRAINING LAYER 3 MULTI-OUTPUT XGBOOST MODEL ---")
saved_model_files = []

for i, col in enumerate(target_cols):
    t_idx = target_indices[i]

    y_train_target = Y_train[:, :, i]  # Shape: (samples, 12)
    y_test_target = Y_test[:, :, i]    # Shape: (samples, 12)

    # Train multi-output XGBoost Regressor
    model = xgb.XGBRegressor(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    model.fit(X_train_flat, y_train_target)

    # Save individual target model JSON
    model_filename = f'xgb_model_{col}.json'
    model.save_model(model_filename)
    saved_model_files.append(model_filename)

    preds_scaled = model.predict(X_test_flat)

    # Unscale predictions back to real physical units
    pred_dummies = np.zeros((preds_scaled.shape[0] * FORECAST_LEN, len(feature_cols)))
    target_dummies = np.zeros((y_test_target.shape[0] * FORECAST_LEN, len(feature_cols)))

    pred_dummies[:, t_idx] = preds_scaled.flatten()
    target_dummies[:, t_idx] = y_test_target.flatten()

    real_pred = scaler.inverse_transform(pred_dummies)[:, t_idx]
    real_target = scaler.inverse_transform(target_dummies)[:, t_idx]

    # Calculate Metrics
    mae = mean_absolute_error(real_target, real_pred)
    rmse = np.sqrt(mean_squared_error(real_target, real_pred))
    target_mean = real_target.mean()
    error_pct = (mae / target_mean) * 100

    print(f"Target Feature: {col}")
    print(f"  MAE : {mae:.3f}")
    print(f"  RMSE: {rmse:.3f}")
    print(f"  Mean Value: {target_mean:.2f}")
    print(f"  Error Percentage: {error_pct:.2f}%\n")

# 6. Automatic Download Trigger
print("Triggering download of model artifacts and scaler...")
for file_path in saved_model_files:
    files.download(file_path)

files.download(scaler_path)

Please upload your un-augmented Spain CSV dataset (without SMOTE):


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (8).csv

--- TRAINING LAYER 3 MULTI-OUTPUT XGBOOST MODEL ---
Target Feature: distance_cm
  MAE : 0.368
  RMSE: 0.724
  Mean Value: 26.59
  Error Percentage: 1.38%

Target Feature: soil_raw
  MAE : 38.402
  RMSE: 122.678
  Mean Value: 3865.71
  Error Percentage: 0.99%

Target Feature: flow_rate_l_min
  MAE : 5.145
  RMSE: 13.091
  Mean Value: 30.31
  Error Percentage: 16.97%

Triggering download of model artifacts and scaler...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**LTSM with ignoring the risk value 3**

In [20]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load Dataset & Filter Layer 2 Classification Errors (flood_risk_level == 3)
print("Please upload your un-augmented Spain CSV dataset (without SMOTE):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
df_clean = df[df['flood_risk_level'] != 3].copy().reset_index(drop=True)

# 2. Feature Engineering
df_clean['tank_fullness'] = 1.0 / (df_clean['distance_cm'] + 1e-5)
df_clean['rain_rolling_sum_6'] = df_clean['rain_raw'].rolling(window=6, min_periods=1).sum()

feature_cols = ['flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage', 'tank_fullness', 'rain_rolling_sum_6']
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

# 3. Normalization
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_clean[feature_cols])

train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

SEQ_LEN = 36
FORECAST_LEN = 12

def create_windowed_numpy(data, target_indices, seq_len=36, forecast_len=12):
    X, Y = [], []
    for i in range(len(data) - seq_len - forecast_len + 1):
        X.append(data[i : i + seq_len])
        Y.append(data[i + seq_len : i + seq_len + forecast_len, target_indices])
    return np.array(X), np.array(Y)

X_train, Y_train = create_windowed_numpy(train_data, target_indices, SEQ_LEN, FORECAST_LEN)
X_test, Y_test = create_windowed_numpy(test_data, target_indices, SEQ_LEN, FORECAST_LEN)

Y_train_flat = Y_train.reshape(Y_train.shape[0], -1)
Y_test_flat = Y_test.reshape(Y_test.shape[0], -1)

# 4. Build Stacked LSTM Architecture
model_lstm = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, len(feature_cols))),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(FORECAST_LEN * len(target_cols))
])

model_lstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
model_lstm.fit(X_train, Y_train_flat, epochs=25, batch_size=64, validation_split=0.1, verbose=1)

# 5. Save Model and Scaler
lstm_model_path = 'stacked_lstm_layer3.keras'
scaler_path = 'scaler_lstm.pkl'

model_lstm.save(lstm_model_path)
joblib.dump(scaler, scaler_path)

# 6. Evaluation
preds_scaled = model_lstm.predict(X_test).reshape(-1, FORECAST_LEN, len(target_cols))

print("\n--- STACKED LSTM EVALUATION RESULTS ---")
for i, col in enumerate(target_cols):
    t_idx = target_indices[i]

    y_pred_flat = preds_scaled[:, :, i].flatten()
    y_true_flat = Y_test[:, :, i].flatten()

    pred_dummies = np.zeros((len(y_pred_flat), len(feature_cols)))
    target_dummies = np.zeros((len(y_true_flat), len(feature_cols)))

    pred_dummies[:, t_idx] = y_pred_flat
    target_dummies[:, t_idx] = y_true_flat

    real_pred = scaler.inverse_transform(pred_dummies)[:, t_idx]
    real_target = scaler.inverse_transform(target_dummies)[:, t_idx]

    mae = mean_absolute_error(real_target, real_pred)
    rmse = np.sqrt(mean_squared_error(real_target, real_pred))
    target_mean = real_target.mean()
    error_pct = (mae / target_mean) * 100

    print(f"Target Feature: {col}")
    print(f"  MAE : {mae:.3f} | RMSE: {rmse:.3f} | Error Pct: {error_pct:.2f}%\n")

# 7. Automatic Download Trigger (Colab)
try:
    from google.colab import files
    print("Downloading LSTM artifacts...")
    files.download(lstm_model_path)
    files.download(scaler_path)
except ImportError:
    print("Models saved locally to current working directory.")

Please upload your un-augmented Spain CSV dataset (without SMOTE):


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (6).csv


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - loss: 0.0617 - val_loss: 0.0025
Epoch 2/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - loss: 0.0191 - val_loss: 5.0760e-04
Epoch 3/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 0.0139 - val_loss: 3.0447e-04
Epoch 4/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0117 - val_loss: 1.5495e-04
Epoch 5/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0105 - val_loss: 1.2598e-04
Epoch 6/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0093 - val_loss: 2.8676e-04
Epoch 7/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0086 - val_loss: 6.7670e-04
Epoch 8/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.0081 - val_loss: 4.7867e-04
Epoch 9/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0076 - val_loss: 3.1957e-04
Epoch 10/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0071 - val_loss: 1.6155e-04
Epoch 11/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0068 - val_loss: 2.2430e-04
Epoch

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**CNN-LTSM hybrid with no risk 3 values**

In [21]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load Dataset & Filter Layer 2 Classification Errors (flood_risk_level == 3)
print("Please upload your un-augmented Spain CSV dataset (without SMOTE):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
df_clean = df[df['flood_risk_level'] != 3].copy().reset_index(drop=True)

# 2. Feature Engineering
df_clean['tank_fullness'] = 1.0 / (df_clean['distance_cm'] + 1e-5)
df_clean['rain_rolling_sum_6'] = df_clean['rain_raw'].rolling(window=6, min_periods=1).sum()

feature_cols = ['flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage', 'tank_fullness', 'rain_rolling_sum_6']
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

# 3. Normalization
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_clean[feature_cols])

train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

SEQ_LEN = 36
FORECAST_LEN = 12

def create_windowed_numpy(data, target_indices, seq_len=36, forecast_len=12):
    X, Y = [], []
    for i in range(len(data) - seq_len - forecast_len + 1):
        X.append(data[i : i + seq_len])
        Y.append(data[i + seq_len : i + seq_len + forecast_len, target_indices])
    return np.array(X), np.array(Y)

X_train, Y_train = create_windowed_numpy(train_data, target_indices, SEQ_LEN, FORECAST_LEN)
X_test, Y_test = create_windowed_numpy(test_data, target_indices, SEQ_LEN, FORECAST_LEN)

Y_train_flat = Y_train.reshape(Y_train.shape[0], -1)
Y_test_flat = Y_test.reshape(Y_test.shape[0], -1)

# 4. Build Hybrid CNN-LSTM Architecture
model_cnn_lstm = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(SEQ_LEN, len(feature_cols))),
    Conv1D(filters=32, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(FORECAST_LEN * len(target_cols))
])

model_cnn_lstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
model_cnn_lstm.fit(X_train, Y_train_flat, epochs=25, batch_size=64, validation_split=0.1, verbose=1)

# 5. Save Model and Scaler
cnn_lstm_model_path = 'hybrid_cnn_lstm_layer3.keras'
scaler_path = 'scaler_cnn_lstm.pkl'

model_cnn_lstm.save(cnn_lstm_model_path)
joblib.dump(scaler, scaler_path)

# 6. Evaluation
preds_scaled_cnn = model_cnn_lstm.predict(X_test).reshape(-1, FORECAST_LEN, len(target_cols))

print("\n--- HYBRID CNN-LSTM EVALUATION RESULTS ---")
for i, col in enumerate(target_cols):
    t_idx = target_indices[i]

    y_pred_flat = preds_scaled_cnn[:, :, i].flatten()
    y_true_flat = Y_test[:, :, i].flatten()

    pred_dummies = np.zeros((len(y_pred_flat), len(feature_cols)))
    target_dummies = np.zeros((len(y_true_flat), len(feature_cols)))

    pred_dummies[:, t_idx] = y_pred_flat
    target_dummies[:, t_idx] = y_true_flat

    real_pred = scaler.inverse_transform(pred_dummies)[:, t_idx]
    real_target = scaler.inverse_transform(target_dummies)[:, t_idx]

    mae = mean_absolute_error(real_target, real_pred)
    rmse = np.sqrt(mean_squared_error(real_target, real_pred))
    target_mean = real_target.mean()
    error_pct = (mae / target_mean) * 100

    print(f"Target Feature: {col}")
    print(f"  MAE : {mae:.3f} | RMSE: {rmse:.3f} | Error Pct: {error_pct:.2f}%\n")

# 7. Automatic Download Trigger (Colab)
try:
    from google.colab import files
    print("Downloading CNN-LSTM artifacts...")
    files.download(cnn_lstm_model_path)
    files.download(scaler_path)
except ImportError:
    print("Models saved locally to current working directory.")

Please upload your un-augmented Spain CSV dataset (without SMOTE):


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (7).csv
Epoch 1/25


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0538 - val_loss: 0.0028
Epoch 2/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0140 - val_loss: 2.9167e-04
Epoch 3/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0105 - val_loss: 2.7824e-04
Epoch 4/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0091 - val_loss: 7.3531e-04
Epoch 5/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0083 - val_loss: 3.6099e-04
Epoch 6/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0077 - val_loss: 1.4849e-04
Epoch 7/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0072 - val_loss: 2.2186e-04
Epoch 8/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0069 - val_loss: 1.6037e-04
Epoch 9/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0065 - val_loss: 1.5319e-04
Epoch 10/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.0063 - val_loss: 1.5066e-04
Epoch 11/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.0059 - val_loss: 2.2509e-04
Epoch 12/25
238/23

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Temporal Transformer Encoder Pipeline**
Transformers capture multi-step context via Multi-Head Self-Attention across temporal sequences.

In [27]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load Dataset & Filter Risk Level 3
print("Please upload your un-augmented Spain CSV dataset (without SMOTE):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
df_clean = df[df['flood_risk_level'] != 3].copy().reset_index(drop=True)

# 2. Feature Engineering
df_clean['tank_fullness'] = 1.0 / (df_clean['distance_cm'] + 1e-5)
df_clean['rain_rolling_sum_6'] = df_clean['rain_raw'].rolling(window=6, min_periods=1).sum()

feature_cols = ['flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage', 'tank_fullness', 'rain_rolling_sum_6']
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

# 3. Scale & Window Data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_clean[feature_cols])

train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

SEQ_LEN, FORECAST_LEN = 36, 12

def create_windowed_numpy(data, target_indices, seq_len=36, forecast_len=12):
    X, Y = [], []
    for i in range(len(data) - seq_len - forecast_len + 1):
        X.append(data[i : i + seq_len])
        Y.append(data[i + seq_len : i + seq_len + forecast_len, target_indices])
    return np.array(X), np.array(Y)

X_train, Y_train = create_windowed_numpy(train_data, target_indices, SEQ_LEN, FORECAST_LEN)
X_test, Y_test = create_windowed_numpy(test_data, target_indices, SEQ_LEN, FORECAST_LEN)
Y_train_flat = Y_train.reshape(Y_train.shape[0], -1)

# 4. Build Transformer Encoder Architecture
def transformer_encoder(inputs, head_size=32, num_heads=2, ff_dim=64, dropout=0.1):
    x = LayerNormalization(epsilon=1e-6)(inputs)
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = Dropout(dropout)(x)
    res = x + inputs

    x = LayerNormalization(epsilon=1e-6)(res)
    x = Dense(ff_dim, activation="relu")(x)
    x = Dropout(dropout)(x)
    x = Dense(inputs.shape[-1])(x)
    return x + res

inputs = Input(shape=(SEQ_LEN, len(feature_cols)))
x = transformer_encoder(inputs, head_size=32, num_heads=4, ff_dim=128, dropout=0.1)
x = GlobalAveragePooling1D()(x)
x = Dense(128, activation="relu")(x)
outputs = Dense(FORECAST_LEN * len(target_cols))(x)

model_transformer = Model(inputs, outputs)
model_transformer.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="mse")
model_transformer.fit(X_train, Y_train_flat, epochs=25, batch_size=64, validation_split=0.1, verbose=1)

# 5. Save & Download Artifacts
model_path = 'transformer_layer3.keras'
scaler_path = 'scaler_transformer.pkl'
model_transformer.save(model_path)
joblib.dump(scaler, scaler_path)

# 6. Evaluate
preds_scaled = model_transformer.predict(X_test).reshape(-1, FORECAST_LEN, len(target_cols))

print("\n--- TRANSFORMER EVALUATION ---")
for i, col in enumerate(target_cols):
    t_idx = target_indices[i]
    y_pred_flat = preds_scaled[:, :, i].flatten()
    y_true_flat = Y_test[:, :, i].flatten()

    pred_dummies = np.zeros((len(y_pred_flat), len(feature_cols)))
    target_dummies = np.zeros((len(y_true_flat), len(feature_cols)))
    pred_dummies[:, t_idx] = y_pred_flat
    target_dummies[:, t_idx] = y_true_flat

    real_pred = scaler.inverse_transform(pred_dummies)[:, t_idx]
    real_target = scaler.inverse_transform(target_dummies)[:, t_idx]

    mae = mean_absolute_error(real_target, real_pred)
    rmse = np.sqrt(mean_squared_error(real_target, real_pred))
    target_mean = real_target.mean()
    print(f"Target Feature: {col}")
    print(f"  MAE : {mae:.3f} | RMSE: {rmse:.3f} | Error Pct: {(mae/target_mean)*100:.2f}%\n")

try:
    from google.colab import files
    files.download(model_path)
    files.download(scaler_path)
except ImportError:
    pass

Please upload your un-augmented Spain CSV dataset (without SMOTE):


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (9).csv
Epoch 1/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 14s 27ms/step - loss: 0.0305 - val_loss: 3.4337e-04
Epoch 2/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 0.0069 - val_loss: 6.5716e-04
Epoch 3/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0059 - val_loss: 3.2471e-04
Epoch 4/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0055 - val_loss: 3.4265e-04
Epoch 5/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0053 - val_loss: 5.9301e-04
Epoch 6/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0052 - val_loss: 2.9744e-04
Epoch 7/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0051 - val_loss: 3.9546e-04
Epoch 8/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0051 - val_loss: 4.2144e-04
Epoch 9/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0050 - val_loss: 2.6144e-04
Epoch 10/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0049 - val_loss: 3.0725e-04
Epoch 11/2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Bidirectional GRU Pipeline**
Bi-GRUs run sequence passes in forward and backward directions, capturing trend transitions faster than standard LSTMs with lower memory overhead on microcontrollers.

In [29]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, GRU, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Please upload your un-augmented Spain CSV dataset (without SMOTE):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
df_clean = df[df['flood_risk_level'] != 3].copy().reset_index(drop=True)

df_clean['tank_fullness'] = 1.0 / (df_clean['distance_cm'] + 1e-5)
df_clean['rain_rolling_sum_6'] = df_clean['rain_raw'].rolling(window=6, min_periods=1).sum()

feature_cols = ['flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage', 'tank_fullness', 'rain_rolling_sum_6']
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_clean[feature_cols])

train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

SEQ_LEN, FORECAST_LEN = 36, 12

def create_windowed_numpy(data, target_indices, seq_len=36, forecast_len=12):
    X, Y = [], []
    for i in range(len(data) - seq_len - forecast_len + 1):
        X.append(data[i : i + seq_len])
        Y.append(data[i + seq_len : i + seq_len + forecast_len, target_indices])
    return np.array(X), np.array(Y)

X_train, Y_train = create_windowed_numpy(train_data, target_indices, SEQ_LEN, FORECAST_LEN)
X_test, Y_test = create_windowed_numpy(test_data, target_indices, SEQ_LEN, FORECAST_LEN)
Y_train_flat = Y_train.reshape(Y_train.shape[0], -1)

# Architecture
model_gru = Sequential([
    Bidirectional(GRU(64, return_sequences=True), input_shape=(SEQ_LEN, len(feature_cols))),
    Dropout(0.2),
    Bidirectional(GRU(32, return_sequences=False)),
    Dropout(0.2),
    Dense(FORECAST_LEN * len(target_cols))
])

model_gru.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
model_gru.fit(X_train, Y_train_flat, epochs=25, batch_size=64, validation_split=0.1, verbose=1)

# Save & Download
model_path = 'bigru_layer3.keras'
scaler_path = 'scaler_bigru.pkl'
model_gru.save(model_path)
joblib.dump(scaler, scaler_path)

preds_scaled = model_gru.predict(X_test).reshape(-1, FORECAST_LEN, len(target_cols))

print("\n--- BI-GRU EVALUATION ---")
for i, col in enumerate(target_cols):
    t_idx = target_indices[i]
    y_pred_flat = preds_scaled[:, :, i].flatten()
    y_true_flat = Y_test[:, :, i].flatten()

    pred_dummies = np.zeros((len(y_pred_flat), len(feature_cols)))
    target_dummies = np.zeros((len(y_true_flat), len(feature_cols)))
    pred_dummies[:, t_idx] = y_pred_flat
    target_dummies[:, t_idx] = y_true_flat

    real_pred = scaler.inverse_transform(pred_dummies)[:, t_idx]
    real_target = scaler.inverse_transform(target_dummies)[:, t_idx]

    mae = mean_absolute_error(real_target, real_pred)
    rmse = np.sqrt(mean_squared_error(real_target, real_pred))
    target_mean = real_target.mean()
    print(f"Target Feature: {col}")
    print(f"  MAE : {mae:.3f} | RMSE: {rmse:.3f} | Error Pct: {(mae/target_mean)*100:.2f}%\n")



Please upload your un-augmented Spain CSV dataset (without SMOTE):


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (10).csv
Epoch 1/25


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


238/238 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 0.0439 - val_loss: 5.4456e-04
Epoch 2/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.0139 - val_loss: 3.6115e-04
Epoch 3/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0107 - val_loss: 2.7873e-04
Epoch 4/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 0.0094 - val_loss: 2.2029e-04
Epoch 5/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 0.0086 - val_loss: 1.2974e-04
Epoch 6/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.0079 - val_loss: 1.0851e-04
Epoch 7/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - loss: 0.0075 - val_loss: 1.8363e-04
Epoch 8/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - loss: 0.0071 - val_loss: 0.0011
Epoch 9/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0068 - val_loss: 2.9020e-04
Epoch 10/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - loss: 0.0066 - val_loss: 1.3771e-04
Epoch 11/25
238/238 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 0.0064 - val_loss: 2.3469e-04
Epoch 12/25


**LightGBM Multi-Output Regressor**
LightGBM uses leaf-wise tree growth, training significantly faster than XGBoost while maintaining high precision on structured tabular time steps.

In [30]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load & Prep
print("Please upload your un-augmented Spain CSV dataset (without SMOTE):")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
df_clean = df[df['flood_risk_level'] != 3].copy().reset_index(drop=True)

df_clean['tank_fullness'] = 1.0 / (df_clean['distance_cm'] + 1e-5)
df_clean['rain_rolling_sum_6'] = df_clean['rain_raw'].rolling(window=6, min_periods=1).sum()

feature_cols = ['flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage', 'tank_fullness', 'rain_rolling_sum_6']
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_clean[feature_cols])

train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

SEQ_LEN, FORECAST_LEN = 36, 12

def create_windowed_numpy(data, target_indices, seq_len=36, forecast_len=12):
    X, Y = [], []
    for i in range(len(data) - seq_len - forecast_len + 1):
        X.append(data[i : i + seq_len])
        Y.append(data[i + seq_len : i + seq_len + forecast_len, target_indices])
    return np.array(X), np.array(Y)

X_train, Y_train = create_windowed_numpy(train_data, target_indices, SEQ_LEN, FORECAST_LEN)
X_test, Y_test = create_windowed_numpy(test_data, target_indices, SEQ_LEN, FORECAST_LEN)

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

scaler_path = 'scaler_lgb.pkl'
joblib.dump(scaler, scaler_path)

print("--- LIGHTGBM EVALUATION ---")
lgb_models = {}

for i, col in enumerate(target_cols):
    t_idx = target_indices[i]
    y_train_target = Y_train[:, :, i]
    y_test_target = Y_test[:, :, i]

    # Train multi-output LightGBM Regressor wrapper
    from sklearn.multioutput import MultiOutputRegressor
    model = MultiOutputRegressor(lgb.LGBMRegressor(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.03,
        random_state=42,
        verbosity=-1
    ))
    model.fit(X_train_flat, y_train_target)

    model_filename = f'lgbm_model_{col}.pkl'
    joblib.dump(model, model_filename)
    lgb_models[col] = model_filename

    preds_scaled = model.predict(X_test_flat)

    pred_dummies = np.zeros((preds_scaled.shape[0] * FORECAST_LEN, len(feature_cols)))
    target_dummies = np.zeros((y_test_target.shape[0] * FORECAST_LEN, len(feature_cols)))
    pred_dummies[:, t_idx] = preds_scaled.flatten()
    target_dummies[:, t_idx] = y_test_target.flatten()

    real_pred = scaler.inverse_transform(pred_dummies)[:, t_idx]
    real_target = scaler.inverse_transform(target_dummies)[:, t_idx]

    mae = mean_absolute_error(real_target, real_pred)
    rmse = np.sqrt(mean_squared_error(real_target, real_pred))
    target_mean = real_target.mean()
    print(f"Target Feature: {col}")
    print(f"  MAE : {mae:.3f} | RMSE: {rmse:.3f} | Error Pct: {(mae/target_mean)*100:.2f}%\n")



Please upload your un-augmented Spain CSV dataset (without SMOTE):


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (11).csv
--- LIGHTGBM EVALUATION ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

Target Feature: distance_cm
  MAE : 0.373 | RMSE: 0.728 | Error Pct: 1.40%



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

Target Feature: soil_raw
  MAE : 41.080 | RMSE: 129.719 | Error Pct: 1.06%



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

Target Feature: flow_rate_l_min
  MAE : 5.120 | RMSE: 12.632 | Error Pct: 16.89%



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [31]:
try:
    from google.colab import files
    for filename in lgb_models.values():
        files.download(filename)
    files.download(scaler_path)
except ImportError:
    pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Trying to figure out which one predicts risk 1 and 2 (critical) xgboost or lightGBM

In [32]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import joblib
from google.colab import files
from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load Dataset & Filter Hardware Errors (Risk Level 3)
print("Please upload your un-augmented Spain CSV dataset:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
df_clean = df[df['flood_risk_level'] != 3].copy().reset_index(drop=True)

# 2. Feature Engineering
df_clean['tank_fullness'] = 1.0 / (df_clean['distance_cm'] + 1e-5)
df_clean['rain_rolling_sum_6'] = df_clean['rain_raw'].rolling(window=6, min_periods=1).sum()

feature_cols = ['flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage', 'tank_fullness', 'rain_rolling_sum_6']
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

# 3. Scaling & Windowing
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_clean[feature_cols])

train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

risk_levels_test = df_clean['flood_risk_level'].values[train_size:]

SEQ_LEN = 36
FORECAST_LEN = 12

X_train, Y_train = [], []
X_test, Y_test = [], []
Y_test_risk = []

for i in range(len(train_data) - SEQ_LEN - FORECAST_LEN + 1):
    X_train.append(train_data[i : i + SEQ_LEN])
    Y_train.append(train_data[i + SEQ_LEN : i + SEQ_LEN + FORECAST_LEN, target_indices])

for i in range(len(test_data) - SEQ_LEN - FORECAST_LEN + 1):
    X_test.append(test_data[i : i + SEQ_LEN])
    Y_test.append(test_data[i + SEQ_LEN : i + SEQ_LEN + FORECAST_LEN, target_indices])
    Y_test_risk.append(risk_levels_test[i + SEQ_LEN : i + SEQ_LEN + FORECAST_LEN])

X_train, Y_train = np.array(X_train), np.array(Y_train)
X_test, Y_test = np.array(X_test), np.array(Y_test)
Y_test_risk = np.array(Y_test_risk).flatten()

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# 4. Train Models
xgb_preds = np.zeros_like(Y_test)
lgb_preds = np.zeros_like(Y_test)

print("\n--- TRAINING BOTH MODELS FOR COMPARISON ---")

for i, col in enumerate(target_cols):
    # XGBoost
    model_xgb = xgb.XGBRegressor(
        n_estimators=150, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, random_state=42
    )
    model_xgb.fit(X_train_flat, Y_train[:, :, i])
    xgb_preds[:, :, i] = model_xgb.predict(X_test_flat)

    # LightGBM
    model_lgb = MultiOutputRegressor(lgb.LGBMRegressor(
        n_estimators=150, max_depth=6, learning_rate=0.03,
        random_state=42, verbosity=-1
    ))
    model_lgb.fit(X_train_flat, Y_train[:, :, i])
    lgb_preds[:, :, i] = model_lgb.predict(X_test_flat)

# 5. Risk-Segmented Evaluation
print("\n" + "="*60)
print("       RISK-SEGMENTED PERFORMANCE BREAKDOWN")
print("="*60)

unique_risks = sorted(list(set(Y_test_risk)))

for i, col in enumerate(target_cols):
    t_idx = target_indices[i]

    y_true_flat = Y_test[:, :, i].flatten()
    xgb_flat = xgb_preds[:, :, i].flatten()
    lgb_flat = lgb_preds[:, :, i].flatten()

    # Unscale predictions
    dummy_true = np.zeros((len(y_true_flat), len(feature_cols)))
    dummy_xgb = np.zeros((len(xgb_flat), len(feature_cols)))
    dummy_lgb = np.zeros((len(lgb_flat), len(feature_cols)))

    dummy_true[:, t_idx] = y_true_flat
    dummy_xgb[:, t_idx] = xgb_flat
    dummy_lgb[:, t_idx] = lgb_flat

    real_true = scaler.inverse_transform(dummy_true)[:, t_idx]
    real_xgb = scaler.inverse_transform(dummy_xgb)[:, t_idx]
    real_lgb = scaler.inverse_transform(dummy_lgb)[:, t_idx]

    print(f"\n>>>> TARGET FEATURE: {col.upper()} <<<<")

    for risk in unique_risks:
        mask = (Y_test_risk == risk)
        if mask.sum() == 0:
            continue

        mae_xgb = mean_absolute_error(real_true[mask], real_xgb[mask])
        mae_lgb = mean_absolute_error(real_true[mask], real_lgb[mask])

        mean_val = real_true[mask].mean()
        err_pct_xgb = (mae_xgb / mean_val) * 100
        err_pct_lgb = (mae_lgb / mean_val) * 100

        winner = "XGBoost" if mae_xgb < mae_lgb else "LightGBM"

        print(f"  [Risk Level {risk}] (Samples: {mask.sum()})")
        print(f"    Mean True Value : {mean_val:.2f}")
        print(f"    XGBoost MAE     : {mae_xgb:.3f} ({err_pct_xgb:.2f}%)")
        print(f"    LightGBM MAE    : {mae_lgb:.3f} ({err_pct_lgb:.2f}%)")
        print(f"    --> Best Model  : {winner}\n")

Please upload your un-augmented Spain CSV dataset:


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (12).csv

--- TRAINING BOTH MODELS FOR COMPARISON ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v


       RISK-SEGMENTED PERFORMANCE BREAKDOWN

>>>> TARGET FEATURE: DISTANCE_CM <<<<
  [Risk Level 0] (Samples: 44088)
    Mean True Value : 28.68
    XGBoost MAE     : 0.321 (1.12%)
    LightGBM MAE    : 0.327 (1.14%)
    --> Best Model  : XGBoost

  [Risk Level 1] (Samples: 5136)
    Mean True Value : 12.35
    XGBoost MAE     : 0.591 (4.79%)
    LightGBM MAE    : 0.614 (4.97%)
    --> Best Model  : XGBoost

  [Risk Level 2] (Samples: 1032)
    Mean True Value : 8.12
    XGBoost MAE     : 1.273 (15.67%)
    LightGBM MAE    : 1.142 (14.06%)
    --> Best Model  : LightGBM


>>>> TARGET FEATURE: SOIL_RAW <<<<
  [Risk Level 0] (Samples: 44088)
    Mean True Value : 4050.00
    XGBoost MAE     : 14.763 (0.36%)
    LightGBM MAE    : 17.042 (0.42%)
    --> Best Model  : XGBoost

  [Risk Level 1] (Samples: 5136)
    Mean True Value : 2567.02
    XGBoost MAE     : 211.466 (8.24%)
    LightGBM MAE    : 216.617 (8.44%)
    --> Best Model  : XGBoost

  [Risk Level 2] (Samples: 1032)
    Mean True

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Part A (Step-by-Step): Pinpoints the step where accuracy degrades. If error doubles after t+6, truncating Layer 3 to a 6-step forecast horizon keeps model outputs reliable for edge decision-making.

Part B (Horizon Zones × Risk Levels): Shows whether LightGBM holds its performance advantage for Risk 2 during immediate steps (t+1 to t+4) versus far steps (t+9 to t+12).

In [33]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from google.colab import files
from sklearn.preprocessing import MinMaxScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error

# 1. Load Data
print("Upload your un-augmented dataset CSV:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
df_clean = df[df['flood_risk_level'] != 3].copy().reset_index(drop=True)

# 2. Features & Scaling
df_clean['tank_fullness'] = 1.0 / (df_clean['distance_cm'] + 1e-5)
df_clean['rain_rolling_sum_6'] = df_clean['rain_raw'].rolling(window=6, min_periods=1).sum()

feature_cols = ['flow_rate_l_min', 'distance_cm', 'rain_raw', 'soil_raw', 'tds_voltage', 'tank_fullness', 'rain_rolling_sum_6']
target_cols = ['distance_cm', 'soil_raw', 'flow_rate_l_min']
target_indices = [feature_cols.index(col) for col in target_cols]

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_clean[feature_cols])

train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

risk_levels_test = df_clean['flood_risk_level'].values[train_size:]

SEQ_LEN, FORECAST_LEN = 36, 12

X_train, Y_train = [], []
X_test, Y_test = [], []
Y_test_risk = []

for i in range(len(train_data) - SEQ_LEN - FORECAST_LEN + 1):
    X_train.append(train_data[i : i + SEQ_LEN])
    Y_train.append(train_data[i + SEQ_LEN : i + SEQ_LEN + FORECAST_LEN, target_indices])

for i in range(len(test_data) - SEQ_LEN - FORECAST_LEN + 1):
    X_test.append(test_data[i : i + SEQ_LEN])
    Y_test.append(test_data[i + SEQ_LEN : i + SEQ_LEN + FORECAST_LEN, target_indices])
    Y_test_risk.append(risk_levels_test[i + SEQ_LEN : i + SEQ_LEN + FORECAST_LEN])

X_train, Y_train = np.array(X_train), np.array(Y_train)
X_test, Y_test = np.array(X_test), np.array(Y_test)
Y_test_risk = np.array(Y_test_risk)

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# 3. Train Both Models
xgb_preds = np.zeros_like(Y_test)
lgb_preds = np.zeros_like(Y_test)

print("\n--- TRAINING XGBOOST & LIGHTGBM ---")

for i, col in enumerate(target_cols):
    # XGBoost
    model_xgb = xgb.XGBRegressor(n_estimators=150, max_depth=6, learning_rate=0.03, random_state=42)
    model_xgb.fit(X_train_flat, Y_train[:, :, i])
    xgb_preds[:, :, i] = model_xgb.predict(X_test_flat)

    # LightGBM
    model_lgb = MultiOutputRegressor(lgb.LGBMRegressor(n_estimators=150, max_depth=6, learning_rate=0.03, random_state=42, verbosity=-1))
    model_lgb.fit(X_train_flat, Y_train[:, :, i])
    lgb_preds[:, :, i] = model_lgb.predict(X_test_flat)

# 4. PART A: Step-by-Step Comparison (t+1 to t+12)
print("\n" + "="*70)
print("       PART A: STEP-BY-STEP (t+1 to t+12) MODEL COMPARISON")
print("="*70)

for i, col in enumerate(target_cols):
    t_idx = target_indices[i]
    print(f"\n>>>> TARGET FEATURE: {col.upper()} <<<<")
    print(f"{'Step':<10}{'XGB MAE':<15}{'LGB MAE':<15}{'XGB Error %':<15}{'LGB Error %':<15}{'Winner':<10}")
    print("-" * 80)

    for step in range(FORECAST_LEN):
        y_true_s = Y_test[:, step, i]
        xgb_s = xgb_preds[:, step, i]
        lgb_s = lgb_preds[:, step, i]

        dummy_true = np.zeros((len(y_true_s), len(feature_cols)))
        dummy_xgb = np.zeros((len(xgb_s), len(feature_cols)))
        dummy_lgb = np.zeros((len(lgb_s), len(feature_cols)))

        dummy_true[:, t_idx] = y_true_s
        dummy_xgb[:, t_idx] = xgb_s
        dummy_lgb[:, t_idx] = lgb_s

        real_true = scaler.inverse_transform(dummy_true)[:, t_idx]
        real_xgb = scaler.inverse_transform(dummy_xgb)[:, t_idx]
        real_lgb = scaler.inverse_transform(dummy_lgb)[:, t_idx]

        mae_xgb = mean_absolute_error(real_true, real_xgb)
        mae_lgb = mean_absolute_error(real_true, real_lgb)

        mean_val = real_true.mean()
        err_xgb = (mae_xgb / mean_val) * 100
        err_lgb = (mae_lgb / mean_val) * 100

        winner = "XGBoost" if mae_xgb < mae_lgb else "LightGBM"

        print(f"t+{step+1:<8}{mae_xgb:<15.3f}{mae_lgb:<15.3f}{err_xgb:<15.2f}%{err_lgb:<15.2f}%{winner:<10}")

# 5. PART B: Forecast Zones x Risk Levels Matrix
print("\n" + "="*70)
print("       PART B: HORIZON ZONES x RISK LEVEL MATRIX")
print("="*70)

zones = {
    'Immediate (t+1 to t+4)': range(0, 4),
    'Mid-Range  (t+5 to t+8)': range(4, 8),
    'Far-Range  (t+9 to t+12)': range(8, 12)
}

unique_risks = sorted(list(set(Y_test_risk.flatten())))

for i, col in enumerate(target_cols):
    t_idx = target_indices[i]
    print(f"\n========================================================")
    print(f">>>> TARGET FEATURE: {col.upper()} <<<<")
    print(f"========================================================")

    for zone_name, step_range in zones.items():
        print(f"\n--- Zone: {zone_name} ---")
        print(f"{'Risk Level':<12}{'Samples':<10}{'XGB MAE':<12}{'LGB MAE':<12}{'XGB Err %':<12}{'LGB Err %':<12}{'Winner':<10}")
        print("-" * 75)

        # Flatten target steps within zone
        zone_y_true = Y_test[:, step_range, i].flatten()
        zone_xgb = xgb_preds[:, step_range, i].flatten()
        zone_lgb = lgb_preds[:, step_range, i].flatten()
        zone_risk = Y_test_risk[:, step_range].flatten()

        # Unscale whole zone block
        dummy_true = np.zeros((len(zone_y_true), len(feature_cols)))
        dummy_xgb = np.zeros((len(zone_xgb), len(feature_cols)))
        dummy_lgb = np.zeros((len(zone_lgb), len(feature_cols)))

        dummy_true[:, t_idx] = zone_y_true
        dummy_xgb[:, t_idx] = zone_xgb
        dummy_lgb[:, t_idx] = zone_lgb

        real_true = scaler.inverse_transform(dummy_true)[:, t_idx]
        real_xgb = scaler.inverse_transform(dummy_xgb)[:, t_idx]
        real_lgb = scaler.inverse_transform(dummy_lgb)[:, t_idx]

        for risk in unique_risks:
            mask = (zone_risk == risk)
            if mask.sum() == 0:
                continue

            mae_xgb = mean_absolute_error(real_true[mask], real_xgb[mask])
            mae_lgb = mean_absolute_error(real_true[mask], real_lgb[mask])

            mean_val = real_true[mask].mean()
            err_xgb = (mae_xgb / mean_val) * 100
            err_lgb = (mae_lgb / mean_val) * 100

            winner = "XGBoost" if mae_xgb < mae_lgb else "LightGBM"

            print(f"Risk {risk:<7}{mask.sum():<10}{mae_xgb:<12.3f}{mae_lgb:<12.3f}{err_xgb:<12.2f}%{err_lgb:<12.2f}%{winner:<10}")

Upload your un-augmented dataset CSV:


Saving segura_processed_training_Cloud.csv to segura_processed_training_Cloud (13).csv

--- TRAINING XGBOOST & LIGHTGBM ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v


       PART A: STEP-BY-STEP (t+1 to t+12) MODEL COMPARISON

>>>> TARGET FEATURE: DISTANCE_CM <<<<
Step      XGB MAE        LGB MAE        XGB Error %    LGB Error %    Winner    
--------------------------------------------------------------------------------
t+1       0.107          0.103          0.40           %0.39           %LightGBM  
t+2       0.143          0.143          0.54           %0.54           %LightGBM  
t+3       0.187          0.188          0.70           %0.71           %XGBoost   
t+4       0.237          0.239          0.89           %0.90           %XGBoost   
t+5       0.288          0.286          1.08           %1.08           %LightGBM  
t+6       0.338          0.340          1.27           %1.28           %XGBoost   
t+7       0.387          0.393          1.46           %1.48           %XGBoost   
t+8       0.435          0.446          1.64           %1.68           %XGBoost   
t+9       0.493          0.498          1.86           %1.87           %XGB

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
